# ------------------------------------------------------------------------

---
# ⚠️ WARNING ⚠️
## **This notebook must be run in Google Colab.** It is used to test and build larger models.
---

# ------------------------------------------------------------------------

In [ ]:

from PIL import Image

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from keras.models import clone_model

from keras import Input, Sequential, layers, applications

SEED = 42
CLASS_NAMES = [
    "Ascaris lumbricoides",
    "Capillaria philippinensis",
    "Enterobius vermicularis",
    "Fasciolopsis buski",
    "Hookworm egg",
    "Hymenolepis diminuta",
    "Hymenolepis nana",
    "Opisthorchis viverrine",
    "Paragonimus spp",
    "Taenia spp. egg",
    "Trichuris trichiura",
]
NUM_CLASSES = len(CLASS_NAMES)
IMG_SIZE = 224

# ---------------------------------------
# Loading the Data
# ---------------------------------------

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
import numpy as np

drive.mount('/content/drive')

# Take the most recently modified crops file on Drive
DRIVE_DIR = Path('/content/drive/MyDrive/egg_hunt')
newest = max(DRIVE_DIR.glob('crops_224_m*_j*.npz'), key=lambda p: p.stat().st_mtime)
print("Using:", newest.name)

Path('/content/data').mkdir(parents=True, exist_ok=True)
CACHE = shutil.copy(newest, '/content/data/')

data = np.load(CACHE)
X_train, y_train, X_test, y_test = data['X_train'], data['y_train'], data['X_test'], data['y_test']
del data
print(X_train.shape, X_train.dtype, y_train.shape)

# ---------------------------------------
# Model Creation
# ---------------------------------------

In [ ]:

# Stratified train/validation split (also shuffles – no extra shuffle needed)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=SEED, stratify=y_train
)
del X_train, y_train          # free ~1.6 GB

pd.DataFrame({"train": y_tr.sum(axis=0).astype(int),
              "val":   y_val.sum(axis=0).astype(int)}, index=CLASS_NAMES)

In [ ]:
from keras import Input
from keras.models import Sequential
from keras.layers import (
    RandomFlip, RandomRotation, RandomZoom, RandomContrast, RandomBrightness,
    Rescaling, Conv2D, BatchNormalization, Activation, MaxPooling2D,
    Flatten, GlobalAveragePooling2D, Dense, Dropout,
)
from keras.applications import EfficientNetB0
from keras.optimizers import Adam

METRICS = ["accuracy", "Recall", "Precision", "AUC"]

def make_callbacks(model_name):
    checkpoint_filepath = f'./models/{model_name}.keras'
    return [
        EarlyStopping(monitor="val_loss", patience=50, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=20, min_lr=1e-7),
        ModelCheckpoint(filepath=checkpoint_filepath, monitor="val_loss", save_best_only=True),
    ]


# ------------------------------------------------------------------------------
# MODEL 1: Baseline CNN (same as test_baseline_cnn, so scores stay comparable)
# ------------------------------------------------------------------------------

def build_model(img_size=IMG_SIZE, num_classes=NUM_CLASSES,
                augment=False, seed=None, lr=1e-3):
    """Raw uint8 images (n, 224, 224, 3) in -> class probabilities (n, 11) out."""
    model = Sequential(name="egg_classifier_cnn")
    model.add(Input(shape=(img_size, img_size, 3)))

    # Augmentation (training only, inactive in predict/evaluate, runs on 0-255)
    if augment:
        model.add(RandomFlip("horizontal_and_vertical", seed=seed))
        model.add(RandomRotation(0.25, fill_mode="constant", seed=seed))
        model.add(RandomZoom(0.1, fill_mode="constant", seed=seed))
        model.add(RandomContrast(0.2, seed=seed))
        model.add(RandomBrightness(0.2, value_range=(0, 255), seed=seed))

    model.add(Rescaling(1.0 / 255))        # crops are uint8 (0-255)

    # Block 1
    model.add(Conv2D(32, kernel_size=3, padding="same", activation="relu"))
    model.add(MaxPooling2D(pool_size=2))

    # Block 2
    model.add(Conv2D(64, kernel_size=3, padding="same", activation="relu"))
    model.add(MaxPooling2D(pool_size=2))

    # Block 3
    model.add(Conv2D(128, kernel_size=3, padding="same", activation="relu"))
    model.add(MaxPooling2D(pool_size=2))

    # Classifier
    model.add(Flatten())
    model.add(Dense(128, activation="relu"))
    model.add(Dropout(0.5))
    model.add(Dense(num_classes, activation="softmax"))

    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss="categorical_crossentropy",   # because y is one-hot
        metrics=METRICS,
    )
    return model


# ------------------------------------------------------------------------------
# MODEL 2: Deeper CNN with BatchNorm + GlobalAveragePooling
# ------------------------------------------------------------------------------

def build_model_global_average(img_size=IMG_SIZE, num_classes=NUM_CLASSES,
                               augment=False, seed=None, lr=1e-3):
    """Raw uint8 images in -> class probabilities out.
    4 conv blocks with BatchNorm, GlobalAveragePooling instead of Flatten."""
    model = Sequential(name="egg_classifier_cnn_gap")
    model.add(Input(shape=(img_size, img_size, 3)))

    # Augmentation (training only, inactive in predict/evaluate, runs on 0-255)
    if augment:
        model.add(RandomFlip("horizontal_and_vertical", seed=seed))
        model.add(RandomRotation(0.25, fill_mode="constant", seed=seed))
        model.add(RandomZoom(0.1, fill_mode="constant", seed=seed))
        model.add(RandomContrast(0.2, seed=seed))
        model.add(RandomBrightness(0.2, value_range=(0, 255), seed=seed))

    model.add(Rescaling(1.0 / 255))

    # Block 1
    model.add(Conv2D(32, kernel_size=3, padding="same", use_bias=False))
    model.add(BatchNormalization())
    model.add(Activation("relu"))
    model.add(MaxPooling2D(pool_size=2))

    # Block 2
    model.add(Conv2D(64, kernel_size=3, padding="same", use_bias=False))
    model.add(BatchNormalization())
    model.add(Activation("relu"))
    model.add(MaxPooling2D(pool_size=2))

    # Block 3
    model.add(Conv2D(128, kernel_size=3, padding="same", use_bias=False))
    model.add(BatchNormalization())
    model.add(Activation("relu"))
    model.add(MaxPooling2D(pool_size=2))

    # Block 4
    model.add(Conv2D(256, kernel_size=3, padding="same", use_bias=False))
    model.add(BatchNormalization())
    model.add(Activation("relu"))
    model.add(MaxPooling2D(pool_size=2))

    # Classifier
    model.add(GlobalAveragePooling2D())    # instead of Flatten -> 12.8M params gone
    model.add(Dropout(0.4))
    model.add(Dense(num_classes, activation="softmax"))

    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss="categorical_crossentropy",
        metrics=METRICS,
    )
    return model


# ------------------------------------------------------------------------------
# MODEL 3: EfficientNetB0 transfer learning
# ------------------------------------------------------------------------------

def build_model_efficientNet(img_size=IMG_SIZE, num_classes=NUM_CLASSES,
                             augment=False, seed=None, lr=1e-3):
    """Raw uint8 images (n, 224, 224, 3) in -> class probabilities out.
    EfficientNet does its own rescaling, so NO Rescaling(1/255) here."""
    base = EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=(img_size, img_size, 3),
    )
    base.trainable = False                 # phase 1: only train the new head

    model = Sequential(name="egg_classifier_efficientnet")
    model.add(Input(shape=(img_size, img_size, 3)))

    # Augmentation (training only, inactive in predict/evaluate, runs on 0-255)
    if augment:
        model.add(RandomFlip("horizontal_and_vertical", seed=seed))
        model.add(RandomRotation(0.25, fill_mode="constant", seed=seed))
        model.add(RandomZoom(0.1, fill_mode="constant", seed=seed))
        model.add(RandomContrast(0.2, seed=seed))
        model.add(RandomBrightness(0.2, value_range=(0, 255), seed=seed))

    # Pretrained feature extractor
    model.add(base)

    # Classifier
    model.add(GlobalAveragePooling2D())
    model.add(Dropout(0.3))
    model.add(Dense(num_classes, activation="softmax"))

    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss="categorical_crossentropy",
        metrics=METRICS,
    )
    return model

# ------------------------------------------------------------------------------
# MODEL 3 – PHASE 2: Fine-tuning (unfreeze the top of EfficientNet)
# ------------------------------------------------------------------------------

def unfreeze_efficientNet(model, n_layers=20, lr=1e-5):
    """Unfreeze the last n_layers of the EfficientNet base for fine-tuning.
    Call this AFTER the head has been trained (phase 1).
    BatchNorm layers stay frozen, otherwise their statistics get destroyed."""
    base = model.get_layer("efficientnetb0")
    base.trainable = True

    for layer in base.layers[:-n_layers]:          # everything except the top
        layer.trainable = False
    for layer in base.layers[-n_layers:]:
        if isinstance(layer, BatchNormalization):  # keep BN frozen
            layer.trainable = False

    # Recompile so the change takes effect – with a tiny learning rate
    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss="categorical_crossentropy",
        metrics=METRICS,
    )
    return model


In [ ]:
models = []

model1 = build_model(augment=True)
model1.summary()
models.append(model1)

model2 = build_model_global_average(augment=True)
model2.summary()
models.append(model2)

model3 = build_model_efficientNet(augment=True)
model3.summary()
models.append(model3)

In [ ]:
histories = []

history1 = model1.fit(
     X_tr, y_tr,
     validation_data=(X_val, y_val),
     epochs=100,
     batch_size=32,
     callbacks=make_callbacks('model1'),
     verbose=2
)
histories.append(history1)

In [ ]:
history2 = model2.fit(
     X_tr, y_tr,
     validation_data=(X_val, y_val),
     epochs=100,
     batch_size=32,
     callbacks=make_callbacks('model2'),
     verbose=2
)
histories.append(history2)

In [ ]:
# Phase 1 of fine tuning
history3 = model3.fit(
     X_tr, y_tr,
     validation_data=(X_val, y_val),
     epochs=100,
     batch_size=32,
     callbacks=make_callbacks('model3'),
     verbose=2
)
histories.append(history3)

In [ ]:
# Phase 2: fine-tune the top of EfficientNet
model3_ft = clone_model(model3)
model3_ft.set_weights(model3.get_weights())     # copy the phase-1 weights
model3_ft = unfreeze_efficientNet(model3_ft, n_layers=20, lr=1e-5)
models.append(model3_ft)

history3_ft = model3_ft.fit(
     X_tr, y_tr,
     validation_data=(X_val, y_val),
     epochs=100,
     batch_size=32,
     callbacks=make_callbacks('model3_ft'),
     verbose=2
)
histories.append(history3_ft)

In [ ]:
# Learning curves
for history in histories:
  fig, axes = plt.subplots(1, 2, figsize=(12, 4))
  for ax, metric in zip(axes, ["loss", "accuracy"]):
      ax.plot(history.history[metric], label="train")
      ax.plot(history.history[f"val_{metric}"], label="validation")
      ax.set_title(metric); ax.set_xlabel("epoch"); ax.legend()
  plt.show()

In [ ]:
# Final evaluation on the test set
for model in models:
  scores = model.evaluate(X_test, y_test, return_dict=True)
  print({k: round(v, 4) for k, v in scores.items()})

  y_true = y_test.argmax(axis=1)
  y_pred = model.predict(X_test).argmax(axis=1)
  print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3))

  # Turn one-hot / probability arrays into class indices
  yt = np.argmax(y_true, axis=1) if y_true.ndim > 1 else y_true
  yp = np.argmax(y_pred, axis=1) if y_pred.ndim > 1 else y_pred

  correct = (yt == yp)
  print(f"Correct: {correct.sum()} / {len(yt)}  ({correct.mean():.1%})")

  # Per class: how many of each true class were predicted correctly
  per_class = pd.DataFrame({"true": yt, "correct": correct}) \
      .groupby("true")["correct"].agg(total="size", correct="sum")
  per_class["accuracy"] = round((per_class["correct"] / per_class["total"])*100, 2)
  print(per_class)

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=CLASS_NAMES,
                                        xticks_rotation=90, normalize="true", values_format=".2f")
plt.show()